# Ex3_EKF-PEM
Estimate histories with a fixed EKF/RTS model, then minimize terminal PEM with L-BFGS.

With `RUN_TRAINING = False`, load the pretrained parameters and display the model. View training history with `tensorboard --logdir runs`.


In [ ]:
from utils import configure_runtime
configure_runtime()
from datetime import datetime
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter
import json
import shutil

from data import evaluation_tensors, load_data
from models import TRUE_PHYSICS_PARAMS, make_vehicle, parameter_dict
from estimation import fixed_histories
from identification import train_ekf_pem
from utils import ROOT, load_car_model_params, load_initialization, save_metrics, show_model


## 1. Configuration

In [ ]:
EXPERIMENT = 3
NAME = "Ex3_EKF-PEM"
RUN_TRAINING = False
SEED = 0
system = dict(nx=3, nu=2, nw=3, N=40, T=0.04,
              E=np.eye(3), C=np.array([[1., 0., 0.], [0., 0., 1.]]),
              integration="rk4", rk4_steps=2, m=790.0, l1=1.248, l2=1.7328)
REFERENCE_DIR = ROOT / "runs" / NAME
OUTPUT_DIR = ROOT / "runs" / "new" / (NAME + "_" + datetime.now().strftime("%Y%m%d-%H%M%S"))
np.random.seed(SEED)
torch.manual_seed(SEED)
CANDIDATES = 1
PARAMETER_PERTURBATION = 0.05
MAX_ITERATIONS, EARLY_STOPPING_PATIENCE = 100, 15

system["hidden_limit"] = 3.0


## 2. Data and initial parameters

In [ ]:
data = load_data(EXPERIMENT)
initialization = load_initialization(NAME)
C = torch.as_tensor(system["C"], dtype=torch.float64)
if EXPERIMENT == 2:
    rng = np.random.default_rng(SEED)
    candidates = TRUE_PHYSICS_PARAMS * (1 + rng.uniform(
        -PARAMETER_PERTURBATION, PARAMETER_PERTURBATION, (CANDIDATES, len(TRUE_PHYSICS_PARAMS))))
    margin = 1e-6 * (initialization["parameter_upper"] - initialization["parameter_lower"])
    candidates = np.clip(candidates, initialization["parameter_lower"] + margin,
                         initialization["parameter_upper"] - margin)
else:
    candidates = initialization["parameter_initial"][None]


## 3. Freeze EKF histories and identify the parameters

In [ ]:
if RUN_TRAINING:
    OUTPUT_DIR.mkdir(parents=True)
    candidate_results = []
    for index, theta in enumerate(candidates):
        candidate_init = dict(initialization, parameter_initial=theta)
        model = make_vehicle(system, candidate_init)
        histories = fixed_histories(model, data)
        tensors = evaluation_tensors(data, histories)
        directory = OUTPUT_DIR / f"candidate_{index:02d}"
        directory.mkdir()
        with SummaryWriter(directory) as writer:
            metrics, history = train_ekf_pem(model, tensors, C, directory, writer,
                                            MAX_ITERATIONS, EARLY_STOPPING_PATIENCE)
        candidate_results.append(dict(candidate=index, **metrics))
        np.savez(directory / "initialization.npz", **candidate_init)
        np.save(directory / "history.npy", history)
    selected = min(candidate_results, key=lambda row: row["val_pem"])
    directory = OUTPUT_DIR / f"candidate_{selected['candidate']:02d}"
    shutil.copy2(directory / "model_params_opt.pth", OUTPUT_DIR / "model_params_opt.pth")
    (OUTPUT_DIR / "selection.json").write_text(json.dumps(candidate_results, indent=2))
    shutil.copy2(directory / "initialization.npz", OUTPUT_DIR / "initialization.npz")
    save_metrics(OUTPUT_DIR, selected["train_pem"], selected["val_pem"], selected["selected_step"])
else:
    model = make_vehicle(system, initialization)
    model.double()
    load_car_model_params(model, REFERENCE_DIR / "model_params_opt.pth")


## 4. Loaded model

In [ ]:
directory = OUTPUT_DIR if RUN_TRAINING else REFERENCE_DIR
if RUN_TRAINING:
    with np.load(directory / "initialization.npz") as saved:
        initialization = {key: saved[key].copy() for key in saved.files}
model = make_vehicle(system, initialization)
model.double()
load_car_model_params(model, directory / "model_params_opt.pth")
model.eval()
show_model(directory, model, parameters=parameter_dict(model))


Checkpoint: runs/Ex3_EKF-PEM/model_params_opt.pth
CarDynamicsPytorch()
Model parameters: 23
Bf       6.4983029
Br       9.1991368
Cf       1.0065397
Cr      0.61668192
Df        3191.478
Dr       2932.9769
Ef      -1.0955253
Er     -0.68299964
Shf   0.0082819291
Shr   0.0067380064
Svf      127.96732
Svr      102.84946
Cm1      3996.3985
Cm2    0.092265929
Cr0      1.2883137
Cr2      1.1995238
Iz       2434.8429
